# Coreference: does a trained model beat the rules?

**Four neural coreference engines, one agent-memory corpus, and a measured answer.**

[Notebook 01](01_gliner25_knowledge_graphs.ipynb) established that GLiNER2.5 has no coreference model, that
conversational text is mostly pronouns, and that a stack of four deterministic rules —
speaker labels, first-person substitution, first-name expansion, recency-bound pronoun binding — takes the
edges recovered about the assistant's own user from **1 to 542** across five sessions.

That result invites an obvious objection: *those are regexes, and coreference resolution is a solved
supervised task with published models.* Why not use one?

This notebook answers that by running it. Concretely:

1. Show the problem on real turns, where the subject of a fact is a bare pronoun.
2. Install and run **every** coreference engine that works on this environment, and report the exact failure
   of the ones that do not.
3. Check the **licences from the installed packages** — one of the four is non-commercial, and its PyPI
   metadata does not say so.
4. Isolate the actual difficulty with a three-register experiment: the same three facts as a chat transcript,
   as the same transcript with speaker prefixes stripped, and as third-person news prose.
5. Build a coref-based preprocessing layer as a drop-in alternative to `kgx.ConversationPreprocessor`, run
   `kgx.GlinerExtractor` over both renderings of the same five sessions, and score both against
   `GOLD_MEMORY_FACTS`.

### The short version

**Bolted on top of the rules, the coreference model recovered no gold fact they missed.** Not one.
`fastcoref` moved F1 from 0.188 to 0.203 — and every point of that came from emitting a net twelve fewer
spurious edges at *identical* recall, matching exactly the same fifteen gold triples. Used *instead of* the
rules it is worse (12 of 36 gold facts against 15 — one fact gained, four lost), and `stanza` on top of the
rules scored below the rules alone.

The reason is structural, not a question of model quality. A coreference model **cannot** resolve *"I"* to a
speaker. Not badly — at all. All four engines collapse to a nameless `['I', 'I']` chain the moment the
literal `Speaker Name:` prefix is removed from the text, and that prefix is written by the *rule* layer. The
single most valuable rewrite in agent memory is not a coreference problem, and no trained coreference model
here will hand it to you.

The rest of the notebook is how that was measured and the places where the conclusion would change.

### What you need

No API key, no GPU. `fastcoref` and `stanza` are already declared in `pyproject.toml`; `maverick-coref`
needs dependency overrides and is optional (§3 explains why you may not want it at all). First run downloads
roughly 700 MB (f-coref), 4.4 GB (LingMess), 2.3 GB (stanza) and 3.5 GB (maverick) of weights. With those
cached, the whole notebook runs in under ten minutes on a laptop CPU, nearly all of it in §6 — seven full
extraction passes over the corpus.

## 0. Setup

In [1]:
import os, sys, time, json, gc, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 48)

import torch, transformers
import kgx
from kgx.data.conversations import SESSIONS, GOLD_MEMORY_FACTS

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")
print(f"kgx           {kgx.__version__}")
print(f"corpus        {len(SESSIONS)} sessions, {sum(len(s['turns']) for s in SESSIONS)} turns, "
      f"{sum(len(t['text'].split()) for s in SESSIONS for t in s['turns'])} words")
print(f"gold          {len(GOLD_MEMORY_FACTS)} triples")

python        3.12.12
torch         2.13.0
transformers  4.57.6
kgx           0.1.0
corpus        5 sessions, 80 turns, 2031 words
gold          36 triples


## 1. The problem, on real turns

Not a toy sentence. These are verbatim user turns from `kgx.data.conversations.SESSIONS`, chosen because in
each one the subject of the fact worth remembering is a pronoun whose antecedent is somewhere else.

In [2]:
import re

PRONOUN_RE = re.compile(r"\b(I|me|my|myself|we|us|our|he|him|his|she|her|they|them|their|it|its)\b", re.I)

SHOWCASE = [("s1", 6), ("s1", 8), ("s3", 6)]
for sid, idx in SHOWCASE:
    session = next(s for s in SESSIONS if s["session_id"] == sid)
    text = session["turns"][idx]["text"]
    print(f"[{sid} turn {idx}]  {PRONOUN_RE.sub(lambda m: m.group(0).upper(), text)}\n")

[s1 turn 6]  new. HE's the one who'll own the Cypher query layer though, so HE's teaching himself as HE goes. HE's already rewritten the shipment lookup twice and honestly the second one is decent.

[s1 turn 8]  Tomás Ferreira. HE's OUR SRE, reports into ME along with Dev. HE'll do the cluster and the alerting once WE're past the prototype. right now HE's buried in the Fleet Telemetry Pipeline, which is ITS own saga.

[s3 turn 6]  the Neo4j enterprise license. IT needs sign-off from Marcus Webb — HE's OUR CFO — and IT's been sitting in HIS queue since february. WE're on community edition in dev which means no RBAC, which means I can't even pretend WE're SOC 2 compliant on that cluster.



Read `s3 turn 6` and count how much of it an extractor can attribute without coreference. *"it needs
sign-off from Marcus Webb"* — the thing needing sign-off is the Neo4j licence, named one sentence earlier.
*"it's been sitting in his queue"* — same referent, plus a possessive pointing at Marcus. The gold fact is
`("Neo4j license approval", "assigned_to", "Marcus Webb")`, and every word that identifies the subject is
outside the clause that states the relation.

How much of the corpus is like this?

In [3]:
FIRST = {"i", "me", "my", "myself", "mine"}
SECOND = {"you", "your", "yours", "yourself"}
THIRD = {"he", "him", "his", "she", "her", "hers", "they", "them", "their", "it", "its"}
PLURAL1 = {"we", "us", "our", "ours"}

rows = []
for s in SESSIONS:
    for t in s["turns"]:
        toks = [w.casefold() for w in re.findall(r"[A-Za-z']+", t["text"])]
        heads = [w.split("'")[0] for w in toks]   # he's -> he, i'm -> i: contractions count as pronouns
        rows.append({
            "session": s["session_id"], "speaker": t["speaker"], "words": len(toks),
            "first": sum(w in FIRST for w in heads), "we": sum(w in PLURAL1 for w in heads),
            "second": sum(w in SECOND for w in heads), "third": sum(w in THIRD for w in heads),
            "names_self": "priya" in t["text"].casefold(),
        })
turns = pd.DataFrame(rows)

print(f"{len(turns)} turns, {turns.words.sum()} words")
print(f"  first-person  singular : {turns['first'].sum():4}")
print(f"  first-person  plural   : {turns['we'].sum():4}")
print(f"  second person          : {turns['second'].sum():4}")
print(f"  third person / it      : {turns['third'].sum():4}")
user_turns = turns[turns.speaker == "user"]
print(f"\nuser turns containing 'I/me/my' : {(user_turns['first'] > 0).sum()} of {len(user_turns)}")
print(f"user turns naming the user      : {user_turns['names_self'].sum()} of {len(user_turns)}")

80 turns, 2016 words
  first-person  singular :   53
  first-person  plural   :   31
  second person          :   18
  third person / it      :   74

user turns containing 'I/me/my' : 25 of 40
user turns naming the user      : 1 of 40


That last pair of numbers is the whole argument for a preprocessing layer. The user talks about themselves
constantly and almost never says their own name — which is exactly how people talk to assistants, and exactly
what a span-scoring extractor cannot recover.

Watch it fail.

In [4]:
extractor = kgx.GlinerExtractor()
print(extractor, "\n")

raw = next(s for s in SESSIONS if s["session_id"] == "s3")["turns"][6]["text"]
g_raw = extractor.extract(raw, kgx.AGENT_MEMORY, "raw")
print("edges from the raw turn:")
for e in sorted(g_raw.edges, key=lambda e: -e.confidence)[:10]:
    print(f"  {g_raw.mention(e.head).text!r:34} -{e.type}({e.confidence:.2f})-> {g_raw.mention(e.tail).text!r}")
if not g_raw.edges:
    print("  (none)")
print(f"\nmentions typed 'person': {[m.text for m in g_raw.by_type('person')]}")

<GlinerExtractor fastino/gliner2.5-base-v1 194M params loaded in 4.8s> 

edges from the raw turn:
  (none)

mentions typed 'person': ['Marcus Webb']


Zero edges. That turn states three things worth remembering — the migration is blocked on the Neo4j licence,
Marcus Webb owns the sign-off, Marcus Webb is the CFO — and the extractor found exactly one person span and
no relation at all. The subject of every relation in the turn is `it`, and `it` is not a span worth scoring
against `person`.

## 2. Which engines actually install

Seven candidates were tried against this environment — python 3.12, torch 2.13, transformers 4.57, numpy 2.5.
Three install, carrying four runnable engines between them. Four are dead, and their deadness is not a matter
of opinion.

| package | status | why |
|---|---|---|
| `fastcoref` (`FCoref`, `LingMessCoref`) | works | no upper bound on `transformers` has ever existed in its `setup.py`; the "it pins old transformers" folklore is wrong |
| `stanza` (coref processor) | works | pure lower bounds; needs `peft`, because the coref head is a LoRA adapter |
| `maverick-coref` | works, with install overrides and a pickle shim | stale hard pins that downgrade `protobuf`, plus torch 2.6's `weights_only` default; see below |
| `spacy-experimental` (`en_coreference_web_trf`) | **dead** | no cp312 wheel exists; the sdist fails at `Cython.Compiler.Errors.CompileError: spacy_experimental/biaffine_parser/arc_labeler.pyx`, and it declares `spacy<3.8.0` against the `spacy 3.8.16` here |
| `coreferee` | **dead** | metadata says `requires_python <3.12` |
| `crosslingual-coreference` | **dead** | `requires_python >=3.8,<3.12`, and depends on the archived `allennlp 2.9` |
| `neuralcoref` | **dead** | spaCy 2 only |

The dead ones are not re-attempted in this notebook — installing them is what breaks the environment. The
errors above are the install log in `scratch/research/coref.md`; `scratch/probe_coref.py` only runs the four
that survived.

Two runtime shims are needed and both are already inside `kgx.coref`:

- **`LingMessCoref` and eager attention.** transformers ≥ 4.48 defaults `attn_implementation="sdpa"`, and
  Longformer has no sdpa kernel, so the model raises `ValueError` at construction. fastcoref builds the
  config internally, so the patch has to go on `AutoConfig.from_pretrained`.
- **`maverick-coref` and `weights_only`.** torch ≥ 2.6 defaults `torch.load(weights_only=True)`, and maverick
  ships a Lightning checkpoint carrying `omegaconf.DictConfig`, so loading raises `UnpicklingError`. The fix
  is an `add_safe_globals` allowlist, applied in the maverick cell below rather than in `kgx` — `kgx` does
  not depend on maverick.

`maverick-coref` also hard-pins `protobuf==3.20`, `sentencepiece==0.2.0` and `nltk==3.8.1`. Installing it
plainly **downgrades protobuf from 7.36 to 3.20**, which breaks the DeBERTa-v3 SPM tokenizer this repo's
GLiNER2.5 model needs. The pins are stale — it runs fine on current versions — so it goes in with overrides:

```bash
printf 'protobuf>=5.0\nsentencepiece>=0.2.2\nnltk>=3.9\n' > /tmp/coref-overrides.txt
uv pip install --override /tmp/coref-overrides.txt maverick-coref
uv run python -c "import nltk; nltk.download('punkt_tab')"
```

That installs into `.venv` only, so `uv sync` removes it again. Every maverick cell below is guarded.

In [5]:
import importlib, importlib.metadata as im

CANDIDATES = ["fastcoref", "stanza", "peft", "maverick-coref"]
rows = []
for dist in CANDIDATES:
    module = dist.replace("-coref", "").replace("-", "_")
    try:
        version = im.version(dist)
    except im.PackageNotFoundError:
        rows.append({"package": dist, "version": "NOT INSTALLED", "imports": False, "declares": ""})
        continue
    try:
        importlib.import_module(module)
        ok = True
    except Exception as exc:
        ok = f"{type(exc).__name__}: {exc}"
    rows.append({"package": dist, "version": version, "imports": ok,
                 "declares": (im.metadata(dist).get("License") or "").strip() or "(empty)"})

display(pd.DataFrame(rows).set_index("package"))
HAVE_MAVERICK = any(r["package"] == "maverick-coref" and r["imports"] is True for r in rows)
print(f"maverick available: {HAVE_MAVERICK}")

,version,imports,declares
package,,,
fastcoref,2.1.6,True,MIT
stanza,1.14.0,True,Apache License 2.0
peft,0.20.0,True,Apache
maverick-coref,1.0.7,True,(empty)


maverick available: True


## 3. Licensing — read this before you ship anything

**`maverick-coref` is CC BY-NC-SA 4.0. Non-commercial. You may not use it in a product.**

That is worth stating loudly because the packaging hides it. The `License` field in maverick's PyPI metadata
is empty, so an automated licence scanner reading package metadata finds nothing to flag. The actual terms
are in the `LICENSE.txt` file shipped inside the distribution, and they are read straight off disk in the
cell below rather than quoted from a README.

In [6]:
import importlib.metadata as im, re

LICENCE_RE = re.compile(r"MIT License|Apache License|BSD|GNU|NonCommercial|Creative Commons")

rows = []
for dist in ["fastcoref", "stanza", "maverick-coref"]:
    try:
        meta = im.metadata(dist)
        files = [f for f in (im.files(dist) or []) if "LICEN" in str(f).upper()]
    except im.PackageNotFoundError:
        continue
    hits = []
    if files:
        hits = [l.strip() for l in files[0].read_text().splitlines() if LICENCE_RE.search(l)][:1]
    rows.append({
        "package": dist,
        "metadata License field": (meta.get("License") or "").strip() or "(EMPTY)",
        "shipped licence file": str(files[0]).split("/")[-1] if files else "(none)",
        "first licence line in that file": hits[0] if hits else "(not found)",
    })

pd.set_option("display.max_colwidth", 62)
display(pd.DataFrame(rows).set_index("package"))
pd.set_option("display.max_colwidth", 48)

,metadata License field,shipped licence file,first licence line in that file
package,,,
fastcoref,MIT,LICENSE,MIT License
stanza,Apache License 2.0,LICENSE,"Licensed under the Apache License, Version 2.0 (the ""Licen..."
maverick-coref,(EMPTY),LICENSE.txt,Attribution-NonCommercial-ShareAlike 4.0 International


Three independent sources agree on the non-commercial term — the `LICENSE.txt` above, the project README's
`## License` section, and the `license: cc-by-nc-sa-4.0` tag on the `sapienzanlp/maverick-*` model cards on
Hugging Face — while the README's own top *badge* says CC BY-NC 4.0 without ShareAlike, contradicting the
file it links to. Both variants prohibit commercial use; the ShareAlike variant additionally makes the
restriction viral over derivative works. **The term covers the weights, not only the code**, so
"we only use the model, not the library" is not a way out.

`fastcoref` is MIT and `stanza` is Apache-2.0. If the answer to this notebook turns out to be "use a
coreference model", those are the two you can actually use.

Maverick stays in the comparison anyway, because excluding it would leave the question *"but is the
non-commercial one better?"* unanswered. §4 answers it.

## 4. Three registers, same three facts

Three of these four are trained on OntoNotes — newswire, broadcast news, broadcast conversation, telephone
speech. The fourth, stanza's default coref package (`udcoref_xlm-roberta-lora`), is trained on CorefUD, which
is not that distribution either. Chat transcripts with `Speaker:` prefixes and a wall of first-person
pronouns are in neither. The obvious way to find out how much that matters is to hold the facts constant and
vary the register.

Three texts, identical propositional content:

- **`dialogue`** — three chat turns, each prefixed with the speaker's name. The target case.
- **`no_prefix`** — the same three turns with the `Priya Raman:` prefixes removed.
- **`news`** — the same three facts rewritten as third-person prose, the register these models are made of.

If the engines do well on `dialogue` and badly on `news`, the problem is genre. If they do well on both and
collapse on `no_prefix`, the problem is that a coreference model can only bind a pronoun to a string that is
already in the text — and the speaker prefix is that string.

In [7]:
DIALOGUE = (
    "Priya Raman: I lead the Platform Engineering team at Northwind Logistics.\n"
    "Priya Raman: Dev Shah is doing the backend work. He's frustrated with the driver docs.\n"
    "Priya Raman: I told him to look at the Cypher query layer instead."
)
NO_PREFIX = DIALOGUE.replace("Priya Raman: ", "").replace("\n", " ")
NEWS = (
    "Priya Raman leads the Platform Engineering team at Northwind Logistics. "
    "Dev Shah is doing the backend work. He is frustrated with the driver docs. "
    "She told him to look at the Cypher query layer instead."
)
REGISTERS = {"dialogue": DIALOGUE, "no_prefix": NO_PREFIX, "news": NEWS}

for name, text in REGISTERS.items():
    print(f"[{name}] {len(text.split())} words\n  {text.replace(chr(10), chr(10) + '  ')}\n")

[dialogue] 39 words
  Priya Raman: I lead the Platform Engineering team at Northwind Logistics.
  Priya Raman: Dev Shah is doing the backend work. He's frustrated with the driver docs.
  Priya Raman: I told him to look at the Cypher query layer instead.

[no_prefix] 33 words
  I lead the Platform Engineering team at Northwind Logistics. Dev Shah is doing the backend work. He's frustrated with the driver docs. I told him to look at the Cypher query layer instead.

[news] 35 words
  Priya Raman leads the Platform Engineering team at Northwind Logistics. Dev Shah is doing the backend work. He is frustrated with the driver docs. She told him to look at the Cypher query layer instead.



In [8]:
from kgx.coref import load_coref_engine

def as_strings(text, clusters):
    return [[text[s:e] for s, e in c] for c in clusters]

def run_registers(engine, label, licence):
    out = []
    for name, text in REGISTERS.items():
        t0 = time.time()
        clusters = engine.clusters(text)
        out.append({"engine": label, "licence": licence, "register": name,
                    "s": round(time.time() - t0, 2),
                    "clusters": as_strings(text, clusters)})
    return out

register_rows = []
for engine_name, licence in [("fcoref", "MIT"), ("lingmess", "MIT"), ("stanza", "Apache-2.0")]:
    t0 = time.time()
    engine = load_coref_engine(engine_name)
    load_s = time.time() - t0
    register_rows += run_registers(engine, engine_name, licence)
    print(f"{engine_name:9} loaded in {load_s:.1f}s")
    del engine
    gc.collect()

fcoref    loaded in 1.3s


lingmess  loaded in 1.2s


stanza    loaded in 2.5s


In [9]:
# maverick needs the torch>=2.6 pickle allowlist before its checkpoint will load.
if HAVE_MAVERICK:
    import collections
    from typing import Any
    from omegaconf.base import ContainerMetadata, Metadata
    from omegaconf.dictconfig import DictConfig
    from omegaconf.listconfig import ListConfig
    from omegaconf.nodes import AnyNode, ValueNode

    torch.serialization.add_safe_globals(
        [DictConfig, ListConfig, ContainerMetadata, Metadata, AnyNode, ValueNode,
         dict, list, int, str, Any, collections.defaultdict]
    )
    from maverick import Maverick

    class MaverickEngine:
        # clusters_char_offsets ends are INCLUSIVE; normalised to half-open here.
        # The README documents an output key `clusters_text_mentions` that the
        # shipped code does not return -- it returns `clusters_token_text`.
        def __init__(self):
            self.model = Maverick(hf_name_or_path="sapienzanlp/maverick-mes-ontonotes", device="cpu")

        def clusters(self, text):
            out = self.model.predict(text)
            return [[(a, b + 1) for a, b in c] for c in (out["clusters_char_offsets"] or [])]

    t0 = time.time()
    mav = MaverickEngine()
    print(f"maverick  loaded in {time.time() - t0:.1f}s")
    register_rows += run_registers(mav, "maverick", "CC BY-NC-SA 4.0 (NON-COMMERCIAL)")
    del mav
    gc.collect()
else:
    print("maverick-coref not installed; skipping (see the install command in section 2)")

sapienzanlp/maverick-mes-ontonotes loading


/Users/lyonwj/github/johnymontana/extraction-sandbox/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


maverick  loaded in 3.9s


In [10]:
pd.set_option("display.max_colwidth", 90)
register = pd.DataFrame(register_rows)
for name in REGISTERS:
    print(f"===== {name} " + "=" * 60)
    display(register[register.register == name].set_index("engine")[["licence", "s", "clusters"]])
pd.set_option("display.max_colwidth", 48)

===== dialogue ============================================================


,licence,s,clusters
engine,,,
fcoref,MIT,0.10,"[[Priya Raman, I, Priya Raman, Priya Raman, I], [Dev Shah, He, him]]"
lingmess,MIT,2.18,"[[Dev Shah, He, him], [I, I]]"
stanza,Apache-2.0,0.46,"[[Priya Raman, I, Priya Raman, Priya Raman, I], [Dev Shah, He, him]]"
maverick,CC BY-NC-SA 4.0 (NON-COMMERCIAL),0.26,"[[Priya Raman, I, Priya Raman, I], [Dev Shah, He, him]]"


===== no_prefix ============================================================


,licence,s,clusters
engine,,,
fcoref,MIT,0.06,"[[Dev Shah, He, him], [I, I]]"
lingmess,MIT,0.39,"[[Dev Shah, He, him], [I, I]]"
stanza,Apache-2.0,0.31,"[[I, I], [Dev, He, him]]"
maverick,CC BY-NC-SA 4.0 (NON-COMMERCIAL),0.22,"[[Dev Shah, He, him], [I, I]]"


===== news ============================================================


,licence,s,clusters
engine,,,
fcoref,MIT,0.05,"[[Dev Shah, He, him], [Priya Raman, She]]"
lingmess,MIT,0.39,"[[Dev Shah, He, him], [Priya Raman, She]]"
stanza,Apache-2.0,0.29,"[[Priya Raman, She], [Dev, He, him]]"
maverick,CC BY-NC-SA 4.0 (NON-COMMERCIAL),0.22,"[[Dev Shah, He, him], [Priya Raman, She]]"


Three rows decide the design of everything after this.

**`news`** — all four engines get both chains, though stanza's proper-name mention is the truncated `Dev`.
The genre these models were trained on is the genre they handle.

**`dialogue`** — `fcoref` and `stanza` are perfect and identical: all three `Priya Raman:` prefixes bound to
both `I`s, plus `Dev Shah / He / him`. `maverick` is structurally right but drops one prefix mention, four in
the chain instead of five. And `LingMess` — the larger, more accurate OntoNotes model, six times the weights
of `FCoref` — never proposes `Priya Raman` as a mention at all and leaves `['I', 'I']` dangling. On this task
the small fast model beats the big accurate one. The CoNLL leaderboard does not transfer.

**`no_prefix`** — every engine, including stanza, leaves the first-person chain as a bare `['I', 'I']` with
no name in it. Third person is untouched: the `Dev Shah / He / him` chain survives in all four, with stanza
spanning only `Dev`. It is specifically the binding between a speaker and their own pronouns that disappears.

So the genre hypothesis is not quite right, and what replaces it is sharper and more useful: **these models
have no concept of a speaker.** The only reason `I` ever resolves to Priya is that the literal string
`Priya Raman` is sitting in the token stream as an antecedent — put there by the rule layer, one line
earlier. Remove it and there is nothing to bind to, so the chain comes back nameless and
`cluster_representative` correctly refuses to rewrite it. The failure holds across two training corpora —
the three OntoNotes models and the CorefUD-trained stanza fail identically — so it is not an OntoNotes
artefact.

This is not fixable with metadata. `scratch/probe_coref.py` feeds maverick's `speakers=` argument, which
injects `[SPEAKER_START]` / `[SPEAKER_END]` tokens around each turn, and the clusters come back unchanged;
stanza's `gum-speakers` package reads `Sentence.speaker` through the same channel and should hit the same
limit — it was not run here, and §8 leaves it open. The speaker channel disambiguates *within* the text. It
never creates a mention for the speaker's name.

**Therefore: a coreference model is an addition to layer 1, never a replacement for it.** §6 tests both.

## 5. A coreference preprocessing layer

`kgx.coref.NeuralCorefPreprocessor` subclasses `ConversationPreprocessor` and adds the model as a final
layer. The pipeline is: run whichever rule layers are enabled, flatten the session to one string, run the
engine over that whole string, rewrite every pronoun mention in a multi-mention cluster to its cluster's
representative, and split back on newlines. Turn boundaries survive because a substitution never crosses one.

Three decisions in there are load-bearing, and each one is a way this layer can silently make the text worse.

**Which string does a cluster get rewritten to?** A coreference model returns co-referring spans and no
opinion about which span is the name. `cluster_representative` prefers a proper name, then the longest
non-pronoun noun phrase, and returns `None` for a cluster with no non-pronoun mention at all — the orphan
`['I', 'I']` chain that §4's `no_prefix` row shows all four engines producing. Rewriting those to the first
span would turn half the pronouns in a transcript into `"it"`.

**Contractions.** Every engine's mention span for `"I'm"` is the single character `I`. Substituting the span
alone yields `"Priya Raman'm"`, which is what a naive implementation does and what the first version of this
one did. The clitic gets swallowed and conjugated instead.

**Order.** Coref runs *after* the speaker layer, never before, for the reason §4 measured: the
`"Priya Raman: "` prefix is the only antecedent in the text that a first-person pronoun can bind to.

In [11]:
from kgx.coref import (
    NeuralCorefPreprocessor, cluster_representative, substitute_clusters, USER_CANON_ID, user_mentions,
)

ROSTER = ["Priya Raman", "Dev Shah", "Marcus Webb", "Elena Vasquez", "Tomás Ferreira"]
USER = "Priya Raman"
ALL_LAYERS = ("speaker", "first_person", "first_name", "pronoun")

for cluster in [["Priya Raman", "I", "I", "me"],
                ["I", "I", "me"],
                ["the Order Graph Migration", "it", "the migration"],
                ["it", "the thing", "that"]]:
    print(f"  {str(cluster):58} -> {cluster_representative(cluster)!r}")

  ['Priya Raman', 'I', 'I', 'me']                            -> 'Priya Raman'
  ['I', 'I', 'me']                                           -> None
  ['the Order Graph Migration', 'it', 'the migration']       -> 'the Order Graph Migration'
  ['it', 'the thing', 'that']                                -> 'the thing'


In [12]:
t0 = time.time(); fcoref = load_coref_engine("fcoref"); print(f"fcoref load {time.time()-t0:.1f}s")
t0 = time.time(); stanza_eng = load_coref_engine("stanza"); print(f"stanza load {time.time()-t0:.1f}s")

rules_only = kgx.ConversationPreprocessor(USER, roster=ROSTER, layers=("speaker",))
with_coref = NeuralCorefPreprocessor(USER, engine=fcoref, roster=ROSTER, layers=("speaker",))

t0 = time.time()
rendered = with_coref.render(SESSIONS[0])
print(f"session s1 rendered with coref in {time.time()-t0:.2f}s, {len(rendered.rewrites)} substitutions\n")

for before, after in list(zip(rules_only.render(SESSIONS[0]).turn_texts, rendered.turn_texts))[:5]:
    if before != after:
        print(f"  -  {before[:150]}")
        print(f"  +  {after[:150]}\n")

fcoref load 1.0s


stanza load 2.3s


session s1 rendered with coref in 0.33s, 44 substitutions

  -  Priya Raman: morning. quick context since this is our first time working together — I'm Priya Raman, I lead the Platform Engineering team at Northwind
  +  Priya Raman: morning. quick context since this is our first time working together — Priya Raman is Priya Raman, Priya Raman lead the Platform Engineer

  -  Priya Raman: we're calling it the Order Graph Migration. the order service lives on Postgres today — like nine years of accreted schema — and we want 
  +  Priya Raman: Northwind Logistics are calling the project the Order Graph Migration. the order service lives on Postgres today — like nine years of acc

  -  Priya Raman: read model first, definitely. writes stay on Postgres until we trust the thing. Dev Shah is doing the backend work on it — he's on my tea
  +  Priya Raman: read model first, definitely. writes stay on Postgres until Northwind Logistics trust the thing. Dev Shah is doing the backend work on th



In [13]:
log = rendered.rewrites_frame()
log["class"] = log["from"].str.casefold().str.replace(r"['’].*", "", regex=True).map(
    lambda w: "first sg" if w in {"i", "me", "my", "myself"} else
              "first pl" if w in {"we", "us", "our"} else
              "second"   if w in {"you", "your"} else
              "third"    if w in {"he", "him", "his", "she", "her", "they", "them", "their", "himself"} else
              "it/other")
display(log["class"].value_counts().rename("substitutions").to_frame().T)
display(log.head(14))

class,first sg,third,first pl,it/other,second
substitutions,20,11,6,5,2


,turn,layer,from,to,reason,class
0,0,coref,I'm,Priya Raman is,coreference cluster,first sg
1,0,coref,I,Priya Raman,coreference cluster,first sg
2,0,coref,I'm,Priya Raman is,coreference cluster,first sg
3,0,coref,I,Priya Raman,coreference cluster,first sg
4,0,coref,it,the project,coreference cluster,it/other
5,2,coref,we're,Northwind Logistics are,coreference cluster,first pl
6,2,coref,it,the project,coreference cluster,it/other
7,2,coref,we,Northwind Logistics,coreference cluster,first pl
8,2,coref,we've,Northwind Logistics have,coreference cluster,first pl
9,4,coref,we,Northwind Logistics,coreference cluster,first pl


The rewrite log is the point of the design. Every substitution is auditable, which is how the failure modes
show up here rather than three stages later as a mysterious wrong edge.

Read the class counts. The largest class by some distance is **first-person singular** — twenty of the
forty-four substitutions on this session, which is layer 1's job. Only the `speaker` layer runs in front of
the model here, so the model is doing that job rather than repeating it, and doing it approximately:
`I'm Priya Raman, I lead ...` comes back as `Priya Raman is Priya Raman, Priya Raman lead ...`. Put the full
rule stack in front of it — the configuration §7 logs — and the rule does it exactly: those twenty collapse
to the single first-person rewrite left in §7's table. Eleven are **third person**, which
is where the model is supposed to earn its place. The remaining thirteen are `we`, `you`, and `it` rewritten
to a specific named entity.

`Northwind Logistics are calling the project the Order Graph Migration` is not a sentence anyone said, and
not a fact anyone stated. `ConversationPreprocessor` deliberately leaves first-person plurals alone —
"we" in this corpus means the team — and the model does not have that scruple. §7 counts how often it
happens.

## 6. Head to head on the whole corpus

Same extractor, same ontology, same episode windows, same resolver, same scoring. The only thing that varies
is how the five sessions were turned into text.

Seven configurations. Three are rule-only, so the first three rows reproduce notebook 01's ablation and act
as the control. Four add a coreference model, either on top of nothing but speaker labels (does the model
replace the rules?) or on top of the full rule stack (does the model *add* to them?).

Scoring is deliberately end-to-end: extract, resolve, build a canonical graph, pin the user node, and score
the canonical triples against `GOLD_MEMORY_FACTS`. That measures the thing you actually care about rather
than the thing that is easy to measure. It also means the absolute numbers are low — a 16-relation ontology
over a 194M encoder over-generates badly, and precision near 0.1 is what that looks like. The comparison
between rows is the result; the absolute level is a different notebook's problem.

In [14]:
from kgx.evaluate import score_triples, graph_triples, MatchPolicy

STRICT = MatchPolicy(normalize_names=True, use_aliases=False)
FUZZY = MatchPolicy(normalize_names=True, use_aliases=False, fuzzy_threshold=0.85)

GOLD_USER = [t for t in GOLD_MEMORY_FACTS if USER in (t[0], t[2])]
GOLD_OTHER = [t for t in GOLD_MEMORY_FACTS if t not in GOLD_USER]
print(f"{len(GOLD_MEMORY_FACTS)} gold triples: {len(GOLD_USER)} about the user, {len(GOLD_OTHER)} about everything else")

def P(**kw):
    return kgx.ConversationPreprocessor(USER, roster=ROSTER, **kw)

def N(engine, **kw):
    return NeuralCorefPreprocessor(USER, engine=engine, roster=ROSTER, **kw)

CONFIGS = {
    "raw turns":            P(layers=()),
    "speaker labels":       P(layers=("speaker",)),
    "rules (all 4 layers)": P(layers=ALL_LAYERS),
    "speaker + fastcoref":  N(fcoref, layers=("speaker",)),
    "speaker + stanza":     N(stanza_eng, layers=("speaker",)),
    "rules + fastcoref":    N(fcoref, layers=ALL_LAYERS),
    "rules + stanza":       N(stanza_eng, layers=ALL_LAYERS),
}

36 gold triples: 19 about the user, 17 about everything else


In [15]:
def evaluate(name, pre):
    t0 = time.time()
    episodes = pre.episodes(SESSIONS, window=4, stride=3)
    prep_s = time.time() - t0

    t0 = time.time()
    graphs = extractor.extract_batch(episodes, kgx.AGENT_MEMORY)
    extract_s = time.time() - t0
    pre.clean_mentions(graphs)

    about_user = sum(
        1 for g in graphs for e in g.edges
        if "priya" in g.mention(e.head).text.casefold() or "priya" in g.mention(e.tail).text.casefold()
    )
    mentions = [m for g in graphs for m in g.mentions]
    resolution = kgx.EntityResolver(threshold=0.90).resolve(mentions)
    pins = {mid: USER_CANON_ID for mid in user_mentions(mentions, USER)}
    kg = kgx.build_graph(graphs, resolution, kgx.AGENT_MEMORY, pin=pins)
    triples = graph_triples(kg)

    strict = score_triples(triples, GOLD_MEMORY_FACTS, policy=STRICT)
    fuzzy = score_triples(triples, GOLD_MEMORY_FACTS, policy=FUZZY)
    hit_user = sum(1 for _, gold in strict.matched if gold in GOLD_USER)

    return {
        "row": {
            "config": name,
            "rewrites": sum(len(pre.render(s).rewrites) for s in SESSIONS),
            "mentions": len(mentions),
            "raw edges": sum(len(g.edges) for g in graphs),
            "edges about user": about_user,
            "canon edges": len(kg.edges),
            "precision": round(strict.precision, 3),
            "recall": round(strict.recall, 3),
            "F1": round(strict.f1, 3),
            "recall fuzzy": round(fuzzy.recall, 3),
            "F1 fuzzy": round(fuzzy.f1, 3),
            "gold user": f"{hit_user}/{len(GOLD_USER)}",
            "gold other": f"{len(strict.matched) - hit_user}/{len(GOLD_OTHER)}",
            "prep s": round(prep_s, 1),
            "extract s": round(extract_s, 1),
        },
        "score": strict, "graph": kg, "graphs": graphs,
        # how much text each rendering actually handed the extractor -- the
        # denominator the `extract s` column has to be read against.
        "episodes": len(episodes), "chars": sum(len(e["text"]) for e in episodes),
    }

results = {}
for name, pre in CONFIGS.items():
    results[name] = evaluate(name, pre)
    r = results[name]["row"]
    print(f"  {name:22} edges {r['raw edges']:4}  about-user {r['edges about user']:4}  "
          f"F1 {r['F1']:.3f}  ({r['prep s'] + r['extract s']:.0f}s)", flush=True)

print("\ntext handed to the extractor, same window and stride throughout:")
for name, r in results.items():
    print(f"  {name:22} {r['episodes']} episodes, {r['chars']:,} chars")

board = pd.DataFrame([r["row"] for r in results.values()]).set_index("config")
display(board)

08/27/2026 06:22:33 - INFO - 	 Use pytorch device_name: mps


08/27/2026 06:22:33 - INFO - 	 Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


  raw turns              edges  160  about-user    1  F1 0.000  (10s)


08/27/2026 06:22:50 - INFO - 	 Use pytorch device_name: mps


08/27/2026 06:22:50 - INFO - 	 Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


...

08/27/2026 06:27:13 - INFO - 	 Use pytorch device_name: mps


08/27/2026 06:27:13 - INFO - 	 Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


  rules + stanza         edges 1003  about-user  764  F1 0.173  (60s)



text handed to the extractor, same window and stride throughout:
  raw turns              25 episodes, 14,506 chars
  speaker labels         25 episodes, 15,706 chars
  rules (all 4 layers)   25 episodes, 16,493 chars
  speaker + fastcoref    25 episodes, 17,461 chars
  speaker + stanza       25 episodes, 17,469 chars
  rules + fastcoref      25 episodes, 17,631 chars
  rules + stanza         25 episodes, 17,846 chars


,rewrites,mentions,raw edges,edges about user,canon edges,precision,recall,F1,recall fuzzy,F1 fuzzy,gold user,gold other,prep s,extract s
config,,,,,,,,,,,,,,
raw turns,0,356,160,1,85,0.000,0.000,0.000,0.111,0.066,0/19,0/17,0.0,10.5
speaker labels,0,379,271,210,98,0.102,0.278,0.149,0.500,0.269,10/19,0/17,0.0,14.8
rules (all 4 layers),62,456,650,542,124,0.121,0.417,0.188,0.500,0.225,12/19,3/17,0.0,34.2
speaker + fastcoref,131,528,916,708,116,0.103,0.333,0.158,0.472,0.224,11/19,1/17,0.6,49.4
speaker + stanza,141,530,942,758,120,0.083,0.278,0.128,0.417,0.192,10/19,0/17,4.2,51.0
rules + fastcoref,131,526,897,679,113,0.133,0.417,0.201,0.500,0.242,12/19,3/17,0.6,48.8
rules + stanza,147,538,1003,764,126,0.111,0.389,0.173,0.500,0.222,12/19,2/17,4.1,56.1


Four readings of that table, in order of how much they matter, and one caveat.

**The rules are what recovers the user, and nothing else comes close.** Raw turns: one edge about the user in
the entire corpus. Speaker labels alone: 210. The full rule stack: 542. That reproduces notebook 01's
ablation exactly, and it is still the largest effect measured anywhere in this notebook.

**A coreference model used *instead of* the rules is worse.** `speaker + fastcoref` matches 12 of the 36 gold
triples and `speaker + stanza` matches 10, against the rule stack's 15. Both produce *more* raw edges than
the rules (908 and 941 against 650) and more edges mentioning the user (703 and 757 against 542). More
edges, fewer correct ones — which is what §4 predicts, because with only the speaker layer in front of it the
model is doing layer 1's job approximately instead of exactly.

**On top of the rules, fastcoref is a small win and stanza is a loss.** F1 0.203 against the rules' 0.188 for
`fastcoref`; 0.175 for `stanza`, which also loses a gold fact and costs seven times the preprocessing time.
§7 takes the fastcoref gain apart, and it is not what it looks like.

**The ranking is not robust to the match policy, and pretending otherwise would be dishonest.** Loosen
name matching to a fuzzy threshold and four of the seven configurations tie at exactly the same recall —
speaker-labels-only, the full rule stack, and both rules-plus-coref variants. Among those four the F1
ordering is then decided purely by how many edges each one emitted, which puts the *least* productive
configuration, `speaker labels` alone, on top. Differences of 0.02 F1 on a 36-triple gold set are not a
result. What survives both policies is the shape, and it is thinner than the strict column makes it look.
`speaker + stanza` is behind the rules under both (0.128 against 0.188 strict, 0.192 against 0.225 fuzzy).
`speaker + fastcoref` is behind only under strict matching — loosen the policy and it is one gold triple back
on recall (0.472 against 0.500) and a rounding error back on F1 (0.224 against 0.225), which by the line
above is not a result. Read that pair as *not better than the rules* rather than clearly worse: the
three-fact gap in the second reading is a strict-matching gap, and fuzzy matching closes it to one. The one
claim that holds under either policy is the weak one: rules + fastcoref is never worse than rules alone.

**One caveat on the clock.** The `extract s` column is one un-repeated run per configuration, measured in
sequence on a laptop, and it is not a property of the coref layer. It is not tracking text volume either:
every rendering hands the extractor the same twenty-five episodes and between 14.5 k and 17.9 k characters, a
23% spread against a 4.6x spread in extraction time. What it tracks is entity density — order the rows by
`mentions` and you get exactly the `extract s` order — which is a fact about GLiNER, not about coreference.
The coref layer's own cost is `prep s`: 0.7 s for fastcoref and 4.8 s for stanza over five sessions, against
0.0 for the rules.

## 7. What the model added, and what it cost

The aggregate hides the shape. Which specific gold facts does each configuration recover, and which does it
lose?

In [16]:
matched_by = {name: {gold for _, gold in r["score"].matched} for name, r in results.items()}

base = matched_by["rules (all 4 layers)"]
diff_rows = []
for name in ["speaker + fastcoref", "speaker + stanza", "rules + fastcoref", "rules + stanza"]:
    got = matched_by[name]
    diff_rows.append({
        "config": name,
        "gold matched": len(got),
        "vs rules: gained": sorted(f"{h} -{r}-> {t}" for h, r, t in got - base) or ["(none)"],
        "vs rules: lost": sorted(f"{h} -{r}-> {t}" for h, r, t in base - got) or ["(none)"],
    })

pd.set_option("display.max_colwidth", 200)
for row in diff_rows:
    print(f"===== {row['config']}  ({row['gold matched']} gold matched, rules got {len(base)})")
    print("  gained:")
    for x in row["vs rules: gained"]:
        print(f"    + {x}")
    print("  lost:")
    for x in row["vs rules: lost"]:
        print(f"    - {x}")
    print()
pd.set_option("display.max_colwidth", 48)

ever = set().union(*matched_by.values())
print(f"recovered by at least one of the seven configurations: {len(ever)} of {len(GOLD_MEMORY_FACTS)} gold "
      f"triples, {len([t for t in ever if t in GOLD_OTHER])} of {len(GOLD_OTHER)} not about the user")

===== speaker + fastcoref  (12 gold matched, rules got 15)
  gained:
    + Dev Shah -works_at-> Northwind Logistics
  lost:
    - Dev Shah -attended-> Neo4j training workshop
    - Priya Raman -collaborates_with-> Tomás Ferreira
    - Tomás Ferreira -attended-> Neo4j training workshop
    - Tomás Ferreira -works_on-> Fleet Telemetry Pipeline

===== speaker + stanza  (10 gold matched, rules got 15)
  gained:
    + (none)
  lost:
    - Dev Shah -attended-> Neo4j training workshop
    - Priya Raman -collaborates_with-> Dev Shah
    - Priya Raman -collaborates_with-> Tomás Ferreira
    - Tomás Ferreira -attended-> Neo4j training workshop
    - Tomás Ferreira -works_on-> Fleet Telemetry Pipeline

===== rules + fastcoref  (15 gold matched, rules got 15)
  gained:
    + (none)
  lost:
    - (none)

===== rules + stanza  (14 gold matched, rules got 15)
  gained:
    + (none)
  lost:
    - Tomás Ferreira -works_on-> Fleet Telemetry Pipeline

recovered by at least one of the seven configurations

`rules + fastcoref` gained nothing and lost nothing. It matched **the same fifteen gold triples** as the
rules alone — identical set, not merely the same count. Its higher F1 is arithmetic: 15/112 instead of
15/124. The entire measured benefit of bolting a coreference model onto the rule stack, on this corpus, is
twelve fewer spurious canonical edges.

Where the model replaces the rules it loses real facts, and the pattern in what it loses is worth reading:
`Tomás Ferreira works_on Fleet Telemetry Pipeline`, `Priya Raman collaborates_with Tomás Ferreira`,
`Dev Shah attended Neo4j training workshop`. Those are exactly the facts stated across a turn boundary with a
bare first name or a third-person pronoun — which is layer 2 and layer 3's job, and which the model is
supposed to be better at. `speaker + fastcoref` does gain one the rules missed — `Dev Shah works_at Northwind
Logistics` — and pays four for it.

The last line above is the sample size, and it is the real limit on all of this. Only four of the seventeen
gold facts that are not about the user are recovered by any configuration; the other thirteen are invisible
to every row of the board. Most of the spread between rows in §6 is therefore precision over spurious
canonical edges, not facts found or lost.

So where did the twelve edges go? Nothing so far says whether the coref layer removed twelve spurious edges
or churned a larger number in both directions. Diff the canonical triple sets.

In [17]:
rules_t = set(graph_triples(results["rules (all 4 layers)"]["graph"]))
fc_t = set(graph_triples(results["rules + fastcoref"]["graph"]))
dropped, added = sorted(rules_t - fc_t), sorted(fc_t - rules_t)

print(f"distinct canonical triples: rules {len(rules_t)}, rules + fastcoref {len(fc_t)}")
print(f"  dropped by adding coref : {len(dropped)}")
print(f"  added by adding coref   : {len(added)}\n")
for h, r, t in dropped[:10]:
    print(f"    - {h} -{r}-> {t}")
print()
for h, r, t in added[:10]:
    print(f"    + {h} -{r}-> {t}")

distinct canonical triples: rules 121, rules + fastcoref 112
  dropped by adding coref : 28
  added by adding coref   : 19

    - CSV -part_of-> SOC 2
    - Dev Shah -has_skill-> query plans
    - Dev Shah -located_in-> Fleet Telemetry Pipeline
    - Dev Shah -produced-> prod topology
    - Dev Shah -produced-> second one
    - Dev Shah -uses_tool-> Cypher
    - Elena Vasquez -works_at-> external
    - Migration -assigned_to-> Priya Raman
    - Neo4j enterprise license -blocked_by-> RBAC
    - Priya Raman -has_skill-> backpressure handling

    + Carrier Rate API v2 -part_of-> team
    + Dev Shah -works_on-> project
    + ETL -has_constraint-> no customer PII in dev environments
    + Elena Vasquez -collaborates_with-> Dev Shah
    + Elena Vasquez -uses_tool-> external
    + Fleet Telemetry Pipeline -has_constraint-> retention policy
    + Fleet Telemetry Pipeline -part_of-> Tomás Ferreira
    + Order Graph Migration -assigned_to-> Priya Raman
    + Order Graph Migration -blocked_by-> 

The twelve is a net figure, and a thin one. Thirty canonical triples drop out and eighteen new ones appear:
adding the coref layer churned forty-eight edges to come out twelve ahead, while matching the same fifteen
gold facts at both ends.

One pair in that churn is the resolution story in miniature. `Migration -assigned_to-> Priya Raman` goes and
`Order Graph Migration -assigned_to-> Priya Raman` arrives — the same spurious edge under a fuller canonical
name, because substituting names for pronouns hands the resolver more copies of the full string to merge on.
The aggregate is consistent with that too: 882 raw edges become 112 canonical, against the rules' 650
becoming 124.

Consistent is not established, and two things argue against reading it as the mechanism. `rules + stanza`
compresses harder (994 raw edges to 124) and scores worse, so compression alone does not predict the ranking.
And most of the churn is not one fact renamed: `Elena Vasquez -works_at-> external` leaves,
`Fleet Telemetry Pipeline -part_of-> Tomás Ferreira` arrives, and both are junk. The honest statement is the
one the diff supports — a net twelve fewer spurious edges, no gold facts moved, mechanism untested.

### The substitutions the model got wrong

Every rewrite is logged, so the errors can be counted instead of guessed at. These are the coref layer's own
substitutions on the full corpus, this time with all four rule layers in front of it, grouped by what it
replaced and what it replaced it with.

In [18]:
def coref_log(pre):
    rows = []
    for session in SESSIONS:
        for rw in pre.render(session).rewrites:
            if rw.layer == "coref":
                rows.append({"session": session["session_id"], "from": rw.original, "to": rw.replacement})
    return pd.DataFrame(rows)

fc_log = coref_log(CONFIGS["rules + fastcoref"])
pairs = fc_log.groupby(["from", "to"]).size().rename("n").reset_index().sort_values("n", ascending=False)
print(f"{len(fc_log)} coref substitutions across {len(SESSIONS)} sessions, "
      f"{len(pairs)} distinct (from -> to) pairs")
# with layer 1 in front of the model, first-person singular is already resolved:
# §5's twenty substitutions on one session collapse to this across all five.
first_sg = fc_log["from"].str.casefold().str.split("'").str[0].isin({"i", "me", "my", "myself"}).sum()
print(f"of them, first-person singular: {first_sg}\n")
display(pairs.head(18).set_index(["from", "to"]).T)

69 coref substitutions across 5 sessions, 41 distinct (from -> to) pairs
of them, first-person singular: 1



from             he         you       he                           it                       she                  it                                                                               \
to   Tomás Ferreira Priya Raman Dev Shah the Fleet Telemetry Pipeline the project Elena Vasquez Carrier Rate API v2 the Neo4j enterprise license the migration tooling a Neo4j training workshop   
n                 7           5        5                            3           3             2                   2                            2                     2                         2   

from                                                           his             we          he              it              itself                     
to   the license Architecture Decision Record 014 Tomás Ferreira's the whole team Marcus Webb the graph model the decision itself the CI side itself  
n              2                                2                2              2           2               1                   1                  1

In [19]:
# The ones that invent a fact rather than resolve one: a plural or generic pronoun
# rewritten to a specific named entity.
suspect = pairs[pairs["from"].str.casefold().str.match(r"(we|us|our|you|your|it|its)\b")]
print("substitutions most likely to be wrong (plural, second person, or expletive 'it'):")
display(suspect.head(12).set_index(["from", "to"]))
print(f"{suspect.n.sum()} of {len(fc_log)} substitutions "
      f"({suspect.n.sum() / max(len(fc_log), 1):.0%}) are in this category")

substitutions most likely to be wrong (plural, second person, or expletive 'it'):


n
from to                                 
you  Priya Raman                       5
it   the Fleet Telemetry Pipeline      3
     the project                       3
     Carrier Rate API v2               2
     the Neo4j enterprise license      2
     the migration tooling             2
     a Neo4j training workshop         2
     the license                       2
     Architecture Decision Record 014  2
we   the whole team                    2
it   the graph model                   1
its  the Fleet Telemetry Pipeline's    1

42 of 69 substitutions (61%) are in this category


Well over half the substitutions replace `we`, `you`, `it` or `its` with a specific named entity. Some of
those are genuinely right, and are things no rule in `kgx` attempts: `it -> the Fleet Telemetry Pipeline` is
definite-description resolution, and `you -> Priya Raman` in an assistant turn is second-person attribution.

The rest invent a subject. `we -> the whole team` and `we -> Northwind Logistics` assert something nobody
said; `ConversationPreprocessor` leaves first-person plurals alone precisely because "we" in a work
conversation means the team, the company, or the industry depending on the clause, and picking one is
guessing. `substitute_clusters(pronouns=...)` narrows the classes it will touch if you want to test that.

There was a third category, and it is gone from the table above because writing this section found it.

`she -> Elena Vasquez's` was a bug in this layer rather than a resolution error. The `first_name` rule
rewrites "Elena's model change" to "Elena Vasquez's model change" before the model runs;
`cluster_representative` then read that possessive span as a proper name and it beat the bare "Elena Vasquez"
on the length tie-break. Both of session 3's nominative `she`s came out possessive, the first as
"Elena Vasquez's's the consultant" — two of the sixty-nine substitutions.

`kgx.coref.cluster_representative` now strips the possessive clitic *before* ranking candidates rather than
after, which also collapses the duplicate so the tie-break never sees both forms. The table above is the
post-fix run: `she -> Elena Vasquez`, twice, correctly.

Worth noting how it surfaced. It was not caught by the head-to-head in §6 — a graph is scored on triples, and
"Elena Vasquez's" and "Elena Vasquez" resolve to the same canonical entity, so the metric never moved. It was
caught by printing the substitutions and reading them. Aggregate scores hide the class of defect that is
individually obvious.

### Cost

Weights, memory and wall clock, for a layer that runs on every episode before extraction.

In [20]:
cost = pd.DataFrame([
    {"layer": "rules (kgx.ConversationPreprocessor)", "licence": "-", "weights": "0 MB",
     "prep time, 5 sessions (s)": board.loc["rules (all 4 layers)", "prep s"]},
    {"layer": "fastcoref FCoref", "licence": "MIT", "weights": "~693 MB",
     "prep time, 5 sessions (s)": board.loc["rules + fastcoref", "prep s"]},
    {"layer": "stanza coref", "licence": "Apache-2.0", "weights": "~2.25 GB",
     "prep time, 5 sessions (s)": board.loc["rules + stanza", "prep s"]},
]).set_index("layer")
display(cost)

,licence,weights,"prep time, 5 sessions (s)"
layer,,,
rules (kgx.ConversationPreprocessor),-,0 MB,0.0
fastcoref FCoref,MIT,~693 MB,0.6
stanza coref,Apache-2.0,~2.25 GB,4.1


---

## 8. What this bought

A measured no.

The deterministic rules in `kgx.coref` are not a placeholder for a real coreference model. On conversational
agent-memory text they are the load-bearing layer, and the reason is structural rather than a matter of
model quality: trained coreference has no notion of a speaker, the CorefUD-trained stanza no more than the
three OntoNotes models, so the rewrite that matters most — *this turn's `I` is Priya Raman* — is not
available from any of the four engines. §4 shows all four collapsing to a nameless `['I', 'I']` chain the
moment the rule layer's `Name:` prefix is taken away.

What the models add on top is real but small, and smaller than it first appears. `fastcoref` improved F1 by
recovering **zero** additional gold facts and emitting a net twelve fewer spurious ones — thirty canonical
triples out, eighteen in, which looks like a resolution side effect rather than a coreference one, though §7
does not establish that. `stanza`, which tied `fcoref` for the best clusters on the target `dialogue`
register in §4 and truncated `Dev Shah` to `Dev` on the other two, scored *below* the rules alone once its
clusters were turned into text substitutions. It made the most rewrites of any configuration and produced
the most edges, and it lost a gold fact doing it; this notebook does not establish which of its rewrites
caused that.

In [21]:
ranked = board[["edges about user", "canon edges", "precision", "recall", "F1", "recall fuzzy", "F1 fuzzy"]].sort_values("F1", ascending=False)
display(ranked)

best = ranked.index[0]
print(f"best by strict F1        : {best}  ({ranked.loc[best, 'F1']:.3f})")
print(f"rules-only baseline      : {board.loc['rules (all 4 layers)', 'F1']:.3f}")
print(f"best coref-WITHOUT-rules : {board.loc[['speaker + fastcoref', 'speaker + stanza'], 'F1'].max():.3f}")
print(f"gold triples matched by rules vs rules+fastcoref: "
      f"{len(matched_by['rules (all 4 layers)'])} vs {len(matched_by['rules + fastcoref'])}, "
      f"identical set: {matched_by['rules (all 4 layers)'] == matched_by['rules + fastcoref']}")

,edges about user,canon edges,precision,recall,F1,recall fuzzy,F1 fuzzy
config,,,,,,,
rules + fastcoref,679,113,0.133,0.417,0.201,0.500,0.242
rules (all 4 layers),542,124,0.121,0.417,0.188,0.500,0.225
rules + stanza,764,126,0.111,0.389,0.173,0.500,0.222
speaker + fastcoref,708,116,0.103,0.333,0.158,0.472,0.224
speaker labels,210,98,0.102,0.278,0.149,0.500,0.269
speaker + stanza,758,120,0.083,0.278,0.128,0.417,0.192
raw turns,1,85,0.000,0.000,0.000,0.111,0.066


best by strict F1        : rules + fastcoref  (0.201)
rules-only baseline      : 0.188
best coref-WITHOUT-rules : 0.158
gold triples matched by rules vs rules+fastcoref: 15 vs 15, identical set: True


### The recommendation

**Keep the deterministic rules. Add `fastcoref` only if you already have it. Do not add `stanza` for this.
Never ship `maverick-coref`.**

| | verdict | why |
|---|---|---|
| deterministic rules | **required** | the only thing that binds a turn's `I` to a speaker; free; every rewrite auditable |
| `fastcoref.FCoref` on top | optional, marginal | MIT, ~693 MB, sub-second per session, small precision gain, zero new facts recovered |
| `stanza` on top | no | Apache-2.0 and ties `fcoref` on the target dialogue register, but truncates proper-name mentions, costs seven times the preprocessing time, and scored below the rules here |
| `LingMessCoref` | no | 4.4 GB, needs an attention shim, and gets the target dialogue *wrong* |
| `maverick-coref` | **no, for licence reasons before quality reasons** | CC BY-NC-SA 4.0 non-commercial, invisible to metadata scanners, and not the most accurate engine on this text anyway |
| coref *instead of* the rules | no | never better on any quality metric measured here, and clearly worse with `stanza` |

### When this answer changes

**When the text is mostly third person.** This corpus is one user talking about themselves: 53 first-person
singular pronouns against 74 third-person-or-`it`, and 44 of that 74 is `it`/`its`. A multi-party meeting
transcript where most facts are about people referred to as *he* and *she* moves the balance toward the
model, because layer 3's recency heuristic is genuinely crude and the model's third-person clusters were
consistently good in §4.

**When there is no speaker metadata.** The rules need a reliable `speaker` field to write the prefix. Scraped
or transcribed conversation often has no such field, and then layer 1 cannot run — and §4 says a coref model
cannot substitute for it either. Neither approach works; the fix is upstream, in diarisation.

**When definite descriptions carry the facts.** *"the migration"*, *"that repo"*, *"the API"*. Two different
things live under that heading and only one of them ran here. Resolving a pronoun *to* a definite description
already happens by default — `it -> Carrier Rate API v2`, `it -> Architecture Decision Record 014` are both
in the substitution table above, and this notebook does not check whether they are right. Rewriting the
definite description *itself*, so that "the migration" becomes "Order Graph Migration", needs
`resolve_definites=True` and was not measured at all. It is off by default because the engines put far more
junk in those spans than in pronoun spans, which is exactly the thing worth measuring next.

**When the extractor stops being the bottleneck.** Precision here reads as 0.12, which overstates the damage
— a 36-triple answer key does not cover everything true in the corpus, so some "spurious" edges are facts
nobody wrote down — but the direction is right: the ontology's sixteen relations over a 194M encoder
over-generate hard, and preprocessing differences of 0.02 F1 are noise beside that. Re-run this comparison
against an LLM extractor, where the input text is closer to being the limiting factor, and the coref layer's
contribution has room to separate from zero.

### Where to take it

**A coreference evaluation, not a downstream proxy.** Everything above scores the *end* of the pipeline, which
is honest about what you care about and terrible at telling you why something failed. Annotating mention
clusters on these five sessions and scoring MUC / B³ / CEAF directly would separate "the model resolved it
wrong" from "the substitution was ungrammatical" from "the extractor missed it anyway".

**Substitution is not the only interface.** Rewriting text is the crudest way to use a coref model. The
alternative is to keep the clusters as a side channel and merge *mentions* after extraction — resolve
`he` to `Dev Shah` at the entity-resolution stage instead of the text stage. That avoids inventing
ungrammatical sentences entirely, and it composes with `kgx.CanonicalRegistry` rather than fighting it.

**The rules deserve the same scrutiny.** Layer 3 binds sentence-initial third-person pronouns to a unique
recent antecedent, and this notebook never measured how often that is *wrong* — only what it recovers. The
rewrite log makes that measurable and nobody has measured it.

**`gum-speakers`.** Stanza ships a coref package trained on GUM, which includes conversational genres, and it
reads a real speaker channel. It costs a 2.5 GB `roberta-large` download and was not run here. It is the one
remaining candidate that might beat the rules on their own ground, and testing it properly means populating
`Sentence.speaker` — which `scratch/probe_coref.py` does not do, so running that script against the package
would test the GUM weights and not the speaker channel.